# Slab2: download regional grids

Download published Slab2 grids, then optionally plot a regional depth map.
Downloading does not require GMT/PyGMT; plotting uses the separate environment
described in the README's **Optional PyGMT environment** section. Run where
`gdmate` is installed (for development, `python -m pip install -e .` from the repository).

Source: [USGS Slab2 grids](https://apps.usgs.gov/shakemap_geodata/slabs/).
Citation: Hayes (2018), [Slab2 data release](https://doi.org/10.5066/F7PV6JNV).
The downloader discovers dated filenames from the USGS listing and preserves them.
The first example uses Tonga–Kermadec (`ker`) depth and dip grids.

Downloads are **off by default**, so opening or testing the notebook does not require
network access. Set `download_enabled = True` below and run all cells to download.
Existing cached files are reused offline unless `overwrite_downloads = True`.


In [ ]:
from pathlib import Path

from gdmate.slab2 import download_slab2

region = "ker"  # USGS region code; e.g. "cas" for Cascadia.
fields = ("depth", "dip")  # Also available: "strike", "uncertainty".
download_enabled = False
overwrite_downloads = False
network_timeout_s = 30

# Keep the default cache in the repository's ignored notebook data directory.
# Replace data_dir with any preferred local path.
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "gdmate").is_dir()
)
data_dir = repo_root / "notebooks/data/slab2"
print(f"Region: {region}; fields: {fields}")
print(f"Download directory: {data_dir}")

## Download and inspect file records

Each successful record includes the source URL, local path, byte count and whether
it came from cache. New downloads use temporary files and are published only after
checking the response length (when supplied) and NetCDF/HDF5 signature.
These checks establish a basic grid-file response; they do not validate the
scientific contents.

If a request fails, the error includes the USGS URL. Check network access and rerun.
Earlier successful files remain cached if a later download fails. Multiple matching
versions produce an error rather than silently selecting one. Cached files do not
check for upstream updates; use `overwrite_downloads = True` to request a refresh.


In [ ]:
downloaded_grids = []
if download_enabled:
    downloaded_grids = download_slab2(
        region, data_dir, fields=fields, overwrite=overwrite_downloads,
        timeout=network_timeout_s,
    )
    for record in downloaded_grids:
        status = "cached" if record["cached"] else "downloaded"
        print(f"{record['field']}: {status}, {record['size_bytes']:,} bytes")
        print(f"  URL:  {record['url']}")
        print(f"  File: {record['path']}")
else:
    print("Downloads disabled. Set download_enabled = True and rerun.")

## Save provenance

A JSON manifest records the source URLs and local files for the next step.
Downloaded files and the manifest remain outside Git in the default cache.


In [ ]:
import json

if downloaded_grids:
    manifest = {
        "citation": "Hayes (2018), https://doi.org/10.5066/F7PV6JNV",
        "region": region,
        "files": [
            dict(record, path=str(record["path"])) for record in downloaded_grids
        ],
    }
    manifest_file = data_dir / f"{region}_download_manifest.json"
    manifest_file.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print(f"Saved manifest: {manifest_file}")

## Plot cached depth with PyGMT

Launch your Jupyter frontend, open this notebook, and select the
**Python (gdmate + PyGMT)** kernel after following the README setup instructions.
Set `plot_enabled = True`. Plotting can run with downloads disabled if the grid
is already cached. No background topography downloads are required.

Native Slab2 depths are negative kilometres below sea level. The map and contours
use **positive-down depth in kilometres**, preserving missing values. The printed
coordinate ranges let you check the longitude convention before changing bounds.

### Plot controls and output

- Leave `depth_grid_file = None` to find the selected region's cached depth grid,
  or provide an explicit path. If multiple versions are cached, choose one explicitly.
- Leave `map_region = None` to use the grid extent, or specify
  `[west, east, south, north]` in degrees using the grid's longitude convention.
- Adjust the projection, colormap and contour/annotation intervals below.
- Inspect the printed coordinates and native depth range before interpreting the map.

PNG and PDF figures are saved under `notebooks/data/slab2/figures/` by default.
The cache, provenance manifest and figures are ignored by Git. Both download and
plot switches default to False, allowing notebook CI without GMT or network access.
Coastlines use GMT's GSHHG data, included by the conda-forge GMT package.

### Seafloor-age background

The combined example places a grayscale seafloor-age grid beneath colored slab
depths, with independent colorbars. Young seafloor is white and old seafloor is
black. This follows the presentation of Zhan's Figure 3b; the age dataset here is
**Seton et al. (2020)** and may differ from the model used in that figure.

Set `download_seafloor_age_enabled = True` for the first run. The
`pygmt.datasets.load_earth_age` call fetches GMT's EarthByte grid and GMT caches
its source data under `~/.gmt/server/earth/earth_age/`. At the default `10m`
resolution GMT downloads the global grid (about 1.3 MB), then crops our region.
The helper also saves a regional NetCDF snapshot under `data_dir` with bounds,
resolution, source citation and retrieval time. Those snapshots are ignored by
Git and are reused without accessing GMT's server. Turn the download switch off
after the first successful run. A changed region or resolution needs its own
snapshot. GMT serves updated datasets over time; retain snapshots for reproducibility.

- `show_seafloor_age = False` restores the slab-only plot.
- `seafloor_age_resolution` controls sampling; default `10m` is 10 arc-minutes.
- `seafloor_age_max_ma` controls the grayscale range (default 0–180 Myr).
  The plotted colorbar uses Ma; ages above the maximum saturate at black.
- Slab and age grids are drawn separately in geographic coordinates; no
  interpolation onto the slab grid is performed by our helper.

References: [GMT EarthByte age dataset](https://www.generic-mapping-tools.org/remote-datasets/earth-age.html),
[Seton et al. (2020)](https://doi.org/10.1029/2020GC009214).


In [ ]:
plot_enabled = False
depth_grid_file = None  # Optional path; otherwise find the region in data_dir.
map_region = None  # Optional [west, east, south, north], using grid longitudes.
map_projection = "M15c"
contour_interval_km = 100
annotation_interval_km = 200
map_colormap = "batlow"
figure_dir = data_dir / "figures"

# Optional seafloor-age background (Myr), with a separate grayscale colorbar.
show_seafloor_age = True
download_seafloor_age_enabled = False  # Enable once to obtain a local snapshot.
seafloor_age_resolution = "10m"  # 10 arc-minutes; suitable for this regional example.
seafloor_age_max_ma = 180  # Older ages saturate at black.


In [ ]:
if plot_enabled:
    from gdmate.slab2 import (
        load_seafloor_age,
        load_slab2_depth,
        plot_slab2_depth,
    )

    if depth_grid_file is None:
        candidates = sorted(data_dir.glob(f"{region}_slab2_dep_*.grd"))
        if len(candidates) != 1:
            raise ValueError(
                f"Expected one cached depth grid, found {len(candidates)}. "
                "Download it above or set depth_grid_file explicitly."
            )
        depth_grid_file = candidates[0]
    depth_grid_file = Path(depth_grid_file)
    depth = load_slab2_depth(depth_grid_file)
    print(f"Grid: {depth_grid_file}")
    print(f"Shape: {depth.shape}; dimensions: {depth.dims}")
    for dimension in depth.dims:
        print(f"{dimension}: {float(depth[dimension].min()):.3f} "
              f"to {float(depth[dimension].max()):.3f} degrees")
    print(f"Native depth range: {-float(depth.max()):.2f} "
          f"to {-float(depth.min()):.2f} km")
    seafloor_age = None
    if show_seafloor_age:
        y_dimension, x_dimension = depth.dims
        age_region = map_region if map_region is not None else [
            float(depth[x_dimension].min()), float(depth[x_dimension].max()),
            float(depth[y_dimension].min()), float(depth[y_dimension].max()),
        ]
        seafloor_age = load_seafloor_age(
            age_region, data_dir, resolution=seafloor_age_resolution,
            download=download_seafloor_age_enabled,
        )
        print(f"Seafloor age: {float(seafloor_age.min()):.2f} "
              f"to {float(seafloor_age.max()):.2f} Myr")
        print(f"Age source: {seafloor_age.attrs['citation']}")
    slab_figure = plot_slab2_depth(
        depth_grid_file, region=map_region, projection=map_projection,
        contour_interval=contour_interval_km,
        annotation_interval=annotation_interval_km, cmap=map_colormap,
        seafloor_age=seafloor_age, age_max=seafloor_age_max_ma,
    )
    figure_dir.mkdir(parents=True, exist_ok=True)
    for suffix in ("png", "pdf"):
        figure_file = figure_dir / f"{region}_slab2_depth.{suffix}"
        slab_figure.savefig(figure_file)
        print(f"Saved: {figure_file}")
    slab_figure.show()
else:
    print("Plot disabled. Set plot_enabled = True in the PyGMT kernel to plot.")

## Download all Slab2 regions

This separate workflow discovers every region with a depth grid in the official
USGS directory and calls the same `download_slab2` downloader for each region.
Depth and dip are requested by default; add `strike` or `uncertainty` to
`global_fields` if needed. This covers grids available from that directory,
not other file types in the complete Slab2 release.

Set `global_download_enabled = True` and run the next cell. Existing cached
files are reused. Each region reports completion or failure, and a manifest is
saved even if a region fails. Rerun to retry failures; the cell raises a summary
error after attempting every region. The global plotting block reads this manifest
without requiring another network request.


In [ ]:
global_download_enabled = False
global_fields = ("depth", "dip")
global_overwrite_downloads = False
global_manifest_file = data_dir / "global_download_manifest.json"

if global_download_enabled:
    import json

    from gdmate.slab2 import list_slab2_regions

    global_regions = list_slab2_regions(timeout=network_timeout_s)
    global_records = []
    global_failures = {}
    for index, code in enumerate(global_regions, start=1):
        try:
            records = download_slab2(
                code, data_dir, fields=global_fields,
                overwrite=global_overwrite_downloads, timeout=network_timeout_s,
            )
            global_records.extend(dict(record, region=code) for record in records)
            print(f"[{index}/{len(global_regions)}] {code}: complete "
                  f"({sum(record['cached'] for record in records)} cached)")
        except (OSError, RuntimeError, ValueError) as error:
            global_failures[code] = str(error)
            print(f"[{index}/{len(global_regions)}] {code}: FAILED: {error}")
    global_manifest = {
        "citation": "Hayes (2018), https://doi.org/10.5066/F7PV6JNV",
        "regions": global_regions,
        "fields": list(global_fields),
        "files": [dict(record, path=str(record["path"])) for record in global_records],
        "failures": global_failures,
    }
    data_dir.mkdir(parents=True, exist_ok=True)
    global_manifest_file.write_text(
        json.dumps(global_manifest, indent=2) + "\n", encoding="utf-8",
    )
    print(f"Saved: {global_manifest_file}")
    if global_failures:
        raise RuntimeError(f"Downloads failed for: {', '.join(global_failures)}")
    print(f"Completed {len(global_regions)} regions, {len(global_records)} files.")
else:
    print("All-region download disabled. Set global_download_enabled = True.")

## Global slab overview

Set `global_plot_enabled = True` after completing the all-region download.
This example plots **slab depths over grayscale age and shaded relief**,
with separate depth and age colorbars and no shoreline outlines by default. It uses a Pacific-centred equidistant cylindrical projection with
latitude bounds of +/-65 degrees. The controls below adjust the centre,
latitude limits, width and colormap. Downloaded grids are overlaid separately;
where they overlap, the later grid in the manifest is drawn on top.

The map requires a complete download manifest and checks each depth file exists.
It reports the number of plotted regions and saves `global_slab2_depth.png` and
`global_slab2_depth.pdf` under `figure_dir`. Slab grids are read from the cache.

Native elevations above sea level (present in a few Muertos nodes) become
negative depths and retain their sign; the map saturates them at the shallow
end of its common color scale.

Set `global_download_age_enabled = True` for the first global age download.
This uses the same GMT EarthByte source and regional-snapshot helper as the
Kermadec example, with the global map bounds. Once cached, set it back to False
for offline plotting. A new centre, latitude range or resolution requires its
own snapshot. No new environment dependencies are needed.

`global_show_seafloor_age = False` restores the slab-only global overview.
`global_age_resolution` defaults to `10m`; `global_age_max_ma` defaults to 180.
Young seafloor is white, old seafloor is black, and missing age data remain white.
The Seton et al. (2020) dataset may differ from the age model in Zhan's Figure 3b.

### Relief without shoreline outlines

Set `global_download_relief_enabled = True` for the first relief download,
then turn it off for offline use. The snapshot uses the same map bounds and
local cache scheme as age. GMT's default **IGPP Earth Relief** dataset provides
land elevation and ocean bathymetry in metres, at `10m` resolution here.
Source URL, source title, bounds, resolution and retrieval time are stored in the
snapshot. See [GMT Earth Relief](https://www.generic-mapping-tools.org/remote-datasets/earth-relief.html).

- `global_show_relief` controls the grayscale elevation base.
- `global_relief_shading` applies relief-derived illumination to terrain
  (light from azimuth 315 degrees). Age is opaque and unshaded wherever present,
  overprinting ocean relief. Land and gaps in age coverage retain shaded relief.
- Age and relief can use different resolutions; elevation and illumination
  are interpolated to the age coordinates for the combined grayscale background.
- `global_draw_shorelines = False` omits coastline outlines. Set it to True
  only if outlines are desired. The regional example keeps its existing outlines.

Slab colors are overlaid last and retain their depth scale. No extra environment
packages are required.

### Tectonic plate boundaries (PB2002)

Set `global_download_boundaries_enabled = True` for the first download of
`PB2002_boundaries.json`, then return it to False to reuse the validated local
snapshot offline. The file is stored under `data_dir`, outside Git, and includes
source URL, original-content SHA256, retrieval time, citation and license metadata.

`global_show_plate_boundaries` toggles the overlay, and `global_boundary_pen`
controls its width/color (default `0.6p,black`). Separate geographic lines and
multiline parts are drawn independently, with unwrapped longitudes for dateline
crossings; GMT clips them at the map seam. The lines are overlaid above slab colors.
These are **tectonic boundaries**, not shorelines; shoreline outlines remain off.
No additional GIS packages are required.

Source: [fraxen/tectonicplates](https://github.com/fraxen/tectonicplates), based on
[Bird (2003), PB2002](https://doi.org/10.1029/2001GC000252). Conversion credits:
Hugo Ahlenius/Nordpil and GeoJSON contributor csterling. The dataset is distributed
under the [Open Data Commons Attribution License](https://opendatacommons.org/licenses/by/1.0/).


In [ ]:
global_plot_enabled = False
global_central_longitude = 180
global_latitude_bounds = (-65, 65)
global_projection_width = "24c"
global_colormap = "batlow"
global_show_seafloor_age = True
global_download_age_enabled = False  # Enable once for this global extent.
global_age_resolution = "10m"
global_age_max_ma = 180
global_show_relief = True
global_download_relief_enabled = False  # Enable once for the relief snapshot.
global_relief_resolution = "10m"
global_relief_shading = True
global_draw_shorelines = False
global_show_plate_boundaries = True
global_download_boundaries_enabled = False  # Enable once to fetch PB2002.
global_boundary_pen = "0.6p,black"

if global_plot_enabled:
    import json

    from gdmate.slab2 import (
        load_earth_relief,
        load_seafloor_age,
        plot_slab2_global,
    )

    global_manifest = json.loads(global_manifest_file.read_text(encoding="utf-8"))
    if global_manifest["failures"]:
        raise ValueError("Complete the failed downloads before plotting.")
    global_depth_records = [
        record for record in global_manifest["files"] if record["field"] == "depth"
    ]
    if {record["region"] for record in global_depth_records} != set(
        global_manifest["regions"]
    ):
        raise ValueError("The manifest does not contain a depth grid for every region.")
    global_map_region = [
        global_central_longitude - 180, global_central_longitude + 180,
        *global_latitude_bounds,
    ]
    global_seafloor_age = None
    if global_show_seafloor_age:
        global_seafloor_age = load_seafloor_age(
            global_map_region, data_dir, resolution=global_age_resolution,
            download=global_download_age_enabled,
        )
        print(f"Global age source: {global_seafloor_age.attrs['citation']}")
    global_relief = None
    if global_show_relief:
        global_relief = load_earth_relief(
            global_map_region, data_dir, resolution=global_relief_resolution,
            download=global_download_relief_enabled,
        )
        print(f"Relief source: {global_relief.attrs['source']}")
    global_plate_boundaries = None
    if global_show_plate_boundaries:
        from gdmate.plate_boundaries import load_plate_boundaries

        global_plate_boundaries = load_plate_boundaries(
            data_dir, download=global_download_boundaries_enabled,
            timeout=network_timeout_s,
        )
        print(f"PB2002: {len(global_plate_boundaries['features'])} boundary features")
    global_figure = plot_slab2_global(
        [record["path"] for record in global_depth_records],
        central_longitude=global_central_longitude,
        latitude_bounds=global_latitude_bounds,
        projection_width=global_projection_width, cmap=global_colormap,
        seafloor_age=global_seafloor_age, age_max=global_age_max_ma,
        relief=global_relief, relief_shading=global_relief_shading,
        shorelines=global_draw_shorelines,
        plate_boundaries=global_plate_boundaries, boundary_pen=global_boundary_pen,
    )
    figure_dir.mkdir(parents=True, exist_ok=True)
    for suffix in ("png", "pdf"):
        global_figure_file = figure_dir / f"global_slab2_depth.{suffix}"
        global_figure.savefig(global_figure_file)
        print(f"Saved: {global_figure_file}")
    print(f"Plotted {len(global_depth_records)} slab regions.")
    global_figure.show()
else:
    print("Global plot disabled. Set global_plot_enabled = True in the PyGMT kernel.")